# Weather Silver Tables

## 1. Tabla: weather_daily_silver

### Descripción
Tabla a nivel **diario** que contiene métricas agregadas del clima por ciudad y fecha.

Cada registro representa:
- 1 ciudad
- 1 día
- Última ingesta disponible (según ingestion_time)

### Granularidad
- city + date

### Columnas

| Columna                  | Tipo      | Descripción |
|--------------------------|----------|-------------|
| city                     | string   | Nombre de la ciudad |
| ingestion_time           | string   | Timestamp de ingesta del dato |
| date                     | date     | Fecha del registro |
| maxtemp_c                | double   | Temperatura máxima del día (°C) |
| mintemp_c                | double   | Temperatura mínima del día (°C) |
| avgtemp_c                | double   | Temperatura promedio del día (°C) |
| avghumidity              | long     | Humedad promedio (%) |
| totalprecip_mm           | double   | Precipitación total (mm) |
| maxwind_kph              | double   | Velocidad máxima del viento (km/h) |
| daily_chance_of_rain     | long     | Probabilidad de lluvia (%) |
| condition                | string   | Condición climática general |

### Lógica aplicada
- Se explota el array `forecastday`
- Se seleccionan métricas del nodo `day`
- Se eliminan duplicados usando:
  - `partitionBy(city, date)`
  - Se conserva el registro con mayor `ingestion_time`

---


In [0]:
from pyspark.sql.functions import col,explode,lower, regexp_replace,desc,row_number
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_history_bronze = spark.read.json("/Volumes/workspace/default/bronce_clima/history/")

In [0]:
df_history_bronze.select("city").distinct().show()

## 2. Tabla: weather_hourly_silver

### Descripción
Tabla a nivel **horario** con métricas detalladas del clima por ciudad, fecha y hora.

Cada registro representa:
- 1 ciudad
- 1 hora específica

### Granularidad
- city + datetime

### Columnas

| Columna        | Tipo      | Descripción |
|----------------|----------|-------------|
| city           | string   | Nombre de la ciudad |
| ingestion_time | string   | Timestamp de ingesta del dato |
| date           | date     | Fecha del registro |
| datetime       | timestamp| Fecha y hora del registro |
| temp_c         | double   | Temperatura (°C) |
| humidity       | long     | Humedad (%) |
| precip_mm      | double   | Precipitación (mm) |
| wind_kph       | double   | Velocidad del viento (km/h) |
| cloud          | long     | Nubosidad (%) |
| condition      | string   | Condición climática |

### Lógica aplicada
- Se explota el array `hour`
- Se aplana la estructura `hour`
- Se renombra:
  - `time` → `datetime`
  - `condition.text` → `condition`
- Se convierte `datetime` a tipo timestamp
- Se eliminan duplicados usando:
  - `partitionBy(city, datetime)`
  - Se conserva el registro con mayor `ingestion_time`

### Notas
- Cada día contiene aproximadamente 24 registros por ciudad
- Todos los registros de un mismo día comparten el mismo `ingestion_time` (por corrida de ingesta)

In [0]:
df_days = df_history_bronze.select(
    col("city"),
    col("metadata.ingestion_time"),
    explode("data.forecast.forecastday").alias("day")
)


In [0]:
df_days.printSchema()

In [0]:
df_daily = df_days.select(
    col("city"),
    col("ingestion_time"),
    col("day.date").alias("date"),
    col("day.day.maxtemp_c").alias("maxtemp_c"),
    col("day.day.mintemp_c").alias("mintemp_c"),
    col("day.day.avgtemp_c").alias("avgtemp_c"),
    col("day.day.avghumidity").alias("avghumidity"),
    col("day.day.totalprecip_mm").alias("totalprecip_mm"),
    col("day.day.maxwind_kph").alias("maxwind_kph"),
    col("day.day.daily_chance_of_rain").alias("daily_chance_of_rain"),
    col("day.day.condition.text").alias("condition")
).withColumn(
    "date", col("date").cast("date")
)

In [0]:
df_daily.show(5)
df_daily.printSchema()


In [0]:
window_spec = Window.partitionBy("city", "date").orderBy(desc("ingestion_time"))

df_daily = df_daily.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [0]:
df_daily.show(10)

In [0]:
df_daily.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_daily_silver")

In [0]:
spark.table("weather_daily_silver").show(10)

### Tabla HOURLY
#### 1 fila = 1 hora ✅


In [0]:
df_hours = df_days.select(
    "city",
    "ingestion_time",
    "day.date",
    explode("day.hour").alias("hour")
)

In [0]:
df_hours.show(5)
df_hours.printSchema()

In [0]:
df_hourly = df_hours.select(
    "city",
    "ingestion_time",
    col("date").cast("date").alias("date"),
    col("hour.time").alias("datetime"),
    col("hour.temp_c"),
    col("hour.humidity"),
    col("hour.precip_mm"),
    col("hour.wind_kph"),
    col("hour.cloud"),
    col("hour.condition.text").alias("condition")
)


In [0]:
df_hourly.show(50)


In [0]:
df_hourly = df_hourly.withColumn(
    "datetime",
    col("datetime").cast("timestamp")
)

In [0]:
window_spec = Window.partitionBy("city", "datetime").orderBy(desc("ingestion_time"))

df_hourly = df_hourly.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [0]:
df_hourly.show(50)

In [0]:
df_hourly.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_hourly_silver")

In [0]:
spark.table("weather_hourly_silver").show(5)